In [ ]:
# Load a file into a dataframe
df = spark.read.load('Files/Data/orders/2019.csv', format='csv', header=False)

# Save the dataframe as a delta table
df.write.format("delta").saveAsTable("mytable")

In [ ]:
df.write.format("delta").saveAsTable("myexternaltable", path="Files/myexternaltable")

In [ ]:
from delta.tables import *

DeltaTable.create(spark) \
  .tableName("products") \
  .addColumn("Productid", "INT") \
  .addColumn("ProductName", "STRING") \
  .addColumn("Category", "STRING") \
  .addColumn("Price", "FLOAT") \
  .execute()

In [ ]:
%%sql

CREATE TABLE salesorders
(
    Orderid INT NOT NULL,
    OrderDate TIMESTAMP NOT NULL,
    CustomerName STRING,
    SalesTotal FLOAT NOT NULL
)
USING DELTA

In [ ]:
%%sql

CREATE TABLE MyExternalTable
USING DELTA
LOCATION 'Files/mydata'

In [ ]:
delta_path = "Files/mydatatable"
df.write.format("delta").save(delta_path)

In [ ]:
new_df.write.format("delta").mode("overwrite").save(delta_path)

In [ ]:
new_rows_df.write.format("delta").mode("append").save(delta_path)

In [ ]:
# Disable Optimize Write at the Spark session level
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", False)

# Enable Optimize Write at the Spark session level
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", True)

print(spark.conf.get("spark.microsoft.delta.optimizeWrite.enabled"))

In [ ]:
%%sql
VACUUM lakehouse2.products RETAIN 168 HOURS;

In [ ]:
%%sql
DESCRIBE HISTORY lakehouse2.products;

In [ ]:
df.write.format("delta").partitionBy("Category").saveAsTable("partitioned_products", path="abfs_path/partitioned_products")

In [ ]:
%%sql
CREATE TABLE partitioned_products (
    ProductID INTEGER,
    ProductName STRING,
    Category STRING,
    ListPrice DOUBLE
)
PARTITIONED BY (Category);

In [ ]:
spark.sql("INSERT INTO products VALUES (1, 'Widget', 'Accessories', 2.99)")

In [ ]:
%%sql

UPDATE products
SET ListPrice = 2.49 WHERE ProductId = 1;

In [ ]:
from delta.tables import *
from pyspark.sql.functions import *

# Create a DeltaTable object
delta_path = "Files/mytable"
deltaTable = DeltaTable.forPath(spark, delta_path)

# Update the table (reduce price of accessories by 10%)
deltaTable.update(
    condition = "Category == 'Accessories'",
    set = { "Price": "Price * 0.9" })

In [ ]:
%%sql

DESCRIBE HISTORY products

In [ ]:
%%sql

DESCRIBE HISTORY 'Files/mytable'

In [ ]:
df = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)


In [ ]:
df = spark.read.format("delta").option("timestampAsOf", '2022-01-01').load(delta_path)

In [ ]:
%%sql
CREATE TABLE orders_in
(
        OrderID INT,
        OrderDate DATE,
        Customer STRING,
        Product STRING,
        Quantity INT,
        Price DECIMAL
)
USING DELTA;

In [ ]:
%%sql
INSERT INTO orders_in (OrderID, OrderDate, Customer, Product, Quantity, Price)
VALUES
    (3001, '2024-09-01', 'Yang', 'Road Bike Red', 1, 1200),
    (3002, '2024-09-01', 'Carlson', 'Mountain Bike Silver', 1, 1500),
    (3003, '2024-09-02', 'Wilson', 'Road Bike Yellow', 2, 1350),
    (3004, '2024-09-02', 'Yang', 'Road Front Wheel', 1, 115),
    (3005, '2024-09-02', 'Rai', 'Mountain Bike Black', 1, NULL);

In [ ]:
# Read and display the input table
df = spark.read.format("delta").table("orders_in")

display(df)

In [ ]:
# Load a streaming DataFrame from the Delta table
stream_df = spark.readStream.format("delta") \
    .option("ignoreChanges", "true") \
    .table("orders_in")

In [ ]:
# Verify that the stream is streaming
stream_df.isStreaming

In [ ]:
from pyspark.sql.functions import col, expr

transformed_df = stream_df.filter(col("Price").isNotNull()) \
    .withColumn('IsBike', expr("INSTR(Product, 'Bike') > 0").cast('int')) \
    .withColumn('Total', expr("Quantity * Price").cast('decimal'))

In [ ]:
# Write the stream to a delta table
output_table_path = 'Tables/orders_processed'
checkpointpath = 'Files/delta/checkpoint'
deltastream = transformed_df.writeStream.format("delta").option("checkpointLocation", checkpointpath).start(output_table_path)

print("Streaming to orders_processed...")

In [ ]:
%%sql
SELECT *
    FROM orders_processed
    ORDER BY OrderID;

In [ ]:
# Stop the streaming data to avoid excessive processing costs
deltastream.stop()